Undersampling + Residual BLock + Sliding Window CNN + LSTM (no Attention)

In [1]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [2]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        # tạo mảng index cho step = 1 (dành riêng cho các class hiếm cần bảo toàn)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        # tạo mảng index theo step được truyền vào (cho các class còn lại)
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.valid_indices = []
        
        # lặp qua từng class (Lấy từ step=1 để đảm bảo không sót class nào)
        classes = np.unique(window_labels_step1)
        rng = np.random.default_rng(seed=42)
        for c in classes:
            # ƯU TIÊN GIỮ NGUYÊN băm cửa sổ 1-1 cho Class 2 và 6
            if c in [100]:
                c_indices = all_indices_step1[window_labels_step1 == c]
            else:
                c_indices = all_indices_stepped[window_labels_stepped == c]
                
            count = len(c_indices)
            
            # nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại
                 
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                if c in [2, 6]:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (sử dụng step=1 để bảo toàn).")
                else:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # lấy các index đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # lấy feature và label của cửa sổ trượt tại index thực sự
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 5 
NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = UndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=1) 
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, set Train Step = 5)...
Đang tính toán các cửa sổ hợp lệ (global step=5) và Undersampling...
Class 0: Giữ nguyên 12897 cửa sổ (step=5).
Class 1: Giảm từ 314644 xuống 20000 cửa sổ.
Class 2: Giữ nguyên 1633 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giảm từ 23563 xuống 20000 cửa sổ.
Class 4: Giữ nguyên 12333 cửa sổ (step=5).
Class 5: Giữ nguyên 1592 cửa sổ (step=5).
Class 6: Giữ nguyên 7698 cửa sổ (sử dụng step=1 để bảo toàn).
Class 7: Giảm từ 69830 xuống 20000 cửa sổ.
Class 8: Giữ nguyên 10884 cửa sổ (step=5).
Class 9: Giảm từ 26987 xuống 20000 cửa sổ.
Class 10: Giữ nguyên 12065 cửa sổ (step=5).
Tổng số cửa sổ sau khi lọc và Undersampling: 139102
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 14880 cửa sổ (step=1).
Class 1: Giữ nguyên 363049 cửa sổ (step=1).
Class 2: Giữ nguyên 1884 cửa sổ (sử dụng step=1 để bảo toàn).
Cla

In [4]:
import torch
import torch.nn as nn

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        # CHỐT CHẶN SE-BLOCK: Lọc bỏ tín hiệu rác do Concept Drift gây ra trên các biến (Features) mỏng manh
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# MODEL CNN-BiLSTM (ĐÃ LƯỢC BỎ ATTENTION)
class CNN_BiLSTM_NoAttention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_NoAttention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        # Đã loại bỏ self.attention
        
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) 
        
        # Đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # CHUẨN HOÁ LAYER NORM THEO TỪNG TIME-STEP
        out = self.layer_norm(out)
        
        # THAY THẾ ATTENTION: Sử dụng Global Average Pooling theo chiều sequence
        # Lấy trung bình tất cả các time-steps để nén về (Batch, hidden_size * 2)
        context_vector = torch.mean(out, dim=1)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

In [5]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_NoAttention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 0.9744, Train Macro F1: 0.7808 | Val Loss: 0.6213, Val Macro F1: 0.8865


Epoch [2/30] - Train Loss: 0.7272, Train Macro F1: 0.9340 | Val Loss: 0.6055, Val Macro F1: 0.8992


Epoch [3/30] - Train Loss: 0.7016, Train Macro F1: 0.9467 | Val Loss: 0.5951, Val Macro F1: 0.9226


Epoch [4/30] - Train Loss: 0.6823, Train Macro F1: 0.9568 | Val Loss: 0.5874, Val Macro F1: 0.9208


...

Epoch [27/30] - Train Loss: 0.6102, Train Macro F1: 0.9893 | Val Loss: 0.5795, Val Macro F1: 0.9182


Epoch [28/30] - Train Loss: 0.6043, Train Macro F1: 0.9923 | Val Loss: 0.5661, Val Macro F1: 0.9418


Epoch [29/30] - Train Loss: 0.6027, Train Macro F1: 0.9922 | Val Loss: 0.5670, Val Macro F1: 0.9445


Epoch [30/30] - Train Loss: 0.5992, Train Macro F1: 0.9931 | Val Loss: 0.5665, Val Macro F1: 0.9448

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.8656    0.8050    0.8342     19846
           1     0.9995    1.0000    0.9998    484077
           2     0.2623    0.9992    0.4155      2515
           3     0.9985    0.7759    0.8732     36253
           4     0.6463    0.8468    0.7331     18979
           5     0.9930    0.9849    0.9889      2451
           6     0.7654    0.6981    0.7302     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.6964    0.9895    0.8175     16746
           9     0.9959    0.7233    0.8380     41514
          10     0.9079    0.9894    0.9469     18567

    accuracy                         0.9600    760231
   macro avg     0.8301    0.8920    0.8343    760231
weighted avg     0.9720    0.9600    0.9625    760231



In [6]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_NoAttention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 0.9894, Train Macro F1: 0.7780 | Val Loss: 0.6375, Val Macro F1: 0.8661


Epoch [2/30] - Train Loss: 0.7311, Train Macro F1: 0.9312 | Val Loss: 0.6087, Val Macro F1: 0.9117


Epoch [3/30] - Train Loss: 0.7102, Train Macro F1: 0.9445 | Val Loss: 0.5933, Val Macro F1: 0.9207


Epoch [4/30] - Train Loss: 0.6918, Train Macro F1: 0.9539 | Val Loss: 0.6299, Val Macro F1: 0.8697


...

Epoch [22/30] - Train Loss: 0.5989, Train Macro F1: 0.9939 | Val Loss: 0.5841, Val Macro F1: 0.9374


Epoch [23/30] - Train Loss: 0.5988, Train Macro F1: 0.9936 | Val Loss: 0.5797, Val Macro F1: 0.9412


Epoch [24/30] - Train Loss: 0.5982, Train Macro F1: 0.9938 | Val Loss: 0.5828, Val Macro F1: 0.9386


Epoch [25/30] - Train Loss: 0.5946, Train Macro F1: 0.9951 | Val Loss: 0.5826, Val Macro F1: 0.9416

[Early Stopping] Model không cải thiện sau 10 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.8703    0.9631    0.9144     19846
           1     0.9989    1.0000    0.9995    484077
           2     0.4421    0.9992    0.6130      2515
           3     0.9990    0.8744    0.9325     36253
           4     0.5437    0.7852    0.6425     18979
           5     0.9917    0.9792    0.9854      2451
           6     0.7778    0.7002    0.7369     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.7773    0.9636    0.8605     16746
           9     0.9958    0.6687    0.8001     41514
          10     0.9375    0.9939    0.9649     18567

    accuracy                         0.9639    760231
   macro avg     0.8486    0.9025    0.8591    760231
weighted avg     0.9725    0.9639    0.9650    760231



UnderSampling + Residual Block + CNN + LSTM + Attention (no Sliding Window)

In [8]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_dataset.X.shape[1]

# CƠ CHẾ ATTENTION THEO THỜI GIAN (Temporal Attention)
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        # CHỐT CHẶN SE-BLOCK: Lọc bỏ tín hiệu rác do Concept Drift gây ra trên các biến (Features) mỏng manh
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# ĐẠI TU MODEL CNN-BiLSTM
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) 
        
        # Đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        if x.size(-1) >=2:
            x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # CHUẨN HOÁ LAYER NORM THEO TỪNG TIME-STEP
        out = self.layer_norm(out)
        
        context_vector, attn_weights = self.attention(out)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out


In [10]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledDataset(Dataset):
    def __init__(self, df, max_samples_per_class=20000):
        # Trích xuất Features và Labels
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        
        print(f"Đang tính toán các mẫu hợp lệ và Undersampling...")
        
        # Mảng chứa tất cả index từ 0 đến len(df) - 1
        all_indices = np.arange(len(self.X))
        all_labels = self.y.numpy()
        
        self.valid_indices = []
        
        # Lặp qua từng class
        classes = np.unique(all_labels)
        rng = np.random.default_rng(seed=42)
        
        for c in classes:
            # Lấy tất cả index của các mẫu thuộc class c
            c_indices = all_indices[all_labels == c]
            count = len(c_indices)
            
            # Nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # Chọn ngẫu nhiên không hoàn lại
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} mẫu.")
            else:
                # Nếu ít hơn hoặc bằng ngưỡng thì giữ nguyên
                print(f"Class {c}: Giữ nguyên {count} mẫu.")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn khi train
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số mẫu sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # Lấy index thực sự đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # Lấy feature và label tại dòng (index) tương ứng
        sample_X = self.X[actual_idx].unsqueeze(0) 
        label_y = self.y[actual_idx]
        
        return sample_X, label_y

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 

BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, đưa mỗi sample riêng lẻ).")
train_dataset = UndersampledDataset(train_df, max_samples_per_class=MAX_SAMPLES)


print(f"Khởi tạo tập Val và Test (Không Undersampling, đưa mỗi sample riêng lẻ, giữ nguyên gốc)...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledDataset(valid_df, max_samples_per_class=10000000)
test_dataset = UndersampledDataset(test_df, max_samples_per_class=10000000)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, đưa mỗi sample riêng lẻ).
Đang tính toán các mẫu hợp lệ và Undersampling...
Class 0: Giảm từ 64488 xuống 20000 mẫu.
Class 1: Giảm từ 1573220 xuống 20000 mẫu.
Class 2: Giữ nguyên 8164 mẫu.
Class 3: Giảm từ 117813 xuống 20000 mẫu.
Class 4: Giảm từ 61666 xuống 20000 mẫu.
Class 5: Giữ nguyên 7956 mẫu.
Class 6: Giảm từ 38490 xuống 20000 mẫu.
Class 7: Giảm từ 349153 xuống 20000 mẫu.
Class 8: Giảm từ 54420 xuống 20000 mẫu.
Class 9: Giảm từ 134940 xuống 20000 mẫu.
Class 10: Giảm từ 60328 xuống 20000 mẫu.
Tổng số mẫu sau khi lọc và Undersampling: 196120
Khởi tạo tập Val và Test (Không Undersampling, đưa mỗi sample riêng lẻ, giữ nguyên gốc)...
Đang tính toán các mẫu hợp lệ và Undersampling...
Class 0: Giữ nguyên 14880 mẫu.
Class 1: Giữ nguyên 363049 mẫu.
Class 2: Giữ nguyên 1884 mẫu.
Class 3: Giữ nguyên 27186 mẫu.
Class 4: Giữ nguyên 14229 mẫu.
Class 5: Giữ nguyên 1836 mẫu.
Class 6: Giữ nguyên 8880 mẫu.
Class 7: Giữ nguyên 80572 mẫu.
Class 8: Giữ nguyên 1255

In [11]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 1.2367, Train Macro F1: 0.6891 | Val Loss: 0.6905, Val Macro F1: 0.7399


Epoch [2/30] - Train Loss: 1.0562, Train Macro F1: 0.7702 | Val Loss: 0.6711, Val Macro F1: 0.7571


Epoch [3/30] - Train Loss: 1.0184, Train Macro F1: 0.7894 | Val Loss: 0.6662, Val Macro F1: 0.7627


Epoch [4/30] - Train Loss: 1.0151, Train Macro F1: 0.7938 | Val Loss: 0.6669, Val Macro F1: 0.7402


...

Epoch [27/30] - Train Loss: 0.8742, Train Macro F1: 0.8875 | Val Loss: 0.6523, Val Macro F1: 0.8428


Epoch [28/30] - Train Loss: 0.8630, Train Macro F1: 0.8926 | Val Loss: 0.6446, Val Macro F1: 0.8145


Epoch [29/30] - Train Loss: 0.8547, Train Macro F1: 0.8931 | Val Loss: 0.6415, Val Macro F1: 0.8164


Epoch [30/30] - Train Loss: 0.8536, Train Macro F1: 0.8925 | Val Loss: 0.6347, Val Macro F1: 0.8432

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.7917    0.6524    0.7153     19846
           1     0.9996    1.0000    0.9998    484077
           2     0.5208    0.8414    0.6434      2515
           3     0.9831    0.9111    0.9457     36253
           4     0.6389    0.8226    0.7192     18979
           5     0.9208    0.9963    0.9571      2451
           6     0.4047    0.7109    0.5158     11847
           7     1.0000    0.9961    0.9980    107436
           8     0.9422    0.8829    0.9116     16746
           9     0.9992    0.6934    0.8187     41523
          10     0.6743    0.8187    0.7395     18567

    accuracy                         0.9529    760240
   macro avg     0.8068    0.8478    0.8149    760240
weighted avg     0.9641    0.9529    0.9555    760240



In [12]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 1.2263, Train Macro F1: 0.6955 | Val Loss: 0.7098, Val Macro F1: 0.6827


Epoch [2/30] - Train Loss: 1.0551, Train Macro F1: 0.7702 | Val Loss: 0.6875, Val Macro F1: 0.7385


Epoch [3/30] - Train Loss: 1.0079, Train Macro F1: 0.7923 | Val Loss: 0.6674, Val Macro F1: 0.7559


Epoch [4/30] - Train Loss: 0.9976, Train Macro F1: 0.7980 | Val Loss: 0.6816, Val Macro F1: 0.7375


...

Epoch [27/30] - Train Loss: 0.8952, Train Macro F1: 0.8470 | Val Loss: 0.6523, Val Macro F1: 0.8207


Epoch [28/30] - Train Loss: 0.8930, Train Macro F1: 0.8606 | Val Loss: 0.6562, Val Macro F1: 0.8237


Epoch [29/30] - Train Loss: 0.9079, Train Macro F1: 0.8694 | Val Loss: 0.6807, Val Macro F1: 0.8206


Epoch [30/30] - Train Loss: 0.9067, Train Macro F1: 0.8853 | Val Loss: 0.6818, Val Macro F1: 0.8501

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.5670    0.2875    0.3815     19846
           1     0.9995    1.0000    0.9998    484077
           2     0.4893    0.7944    0.6056      2515
           3     0.9802    0.8881    0.9319     36253
           4     0.6038    0.4421    0.5104     18979
           5     0.9215    0.9963    0.9575      2451
           6     0.4382    0.7285    0.5472     11847
           7     1.0000    0.9963    0.9981    107436
           8     0.4413    0.8910    0.5902     16746
           9     0.9999    0.6915    0.8176     41523
          10     0.6609    0.8232    0.7332     18567

    accuracy                         0.9331    760240
   macro avg     0.7365    0.7763    0.7339    760240
weighted avg     0.9463    0.9331    0.9340    760240



Dynamic Undersampling + Residual Block + CNN + LSTM + Attention + Sliding Window

In [13]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F 
NUM_FEATURES = train_dataset.X.shape[1]

# CƠ CHẾ ATTENTION THEO THỜI GIAN (Temporal Attention)
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        # CHỐT CHẶN SE-BLOCK: Lọc bỏ tín hiệu rác do Concept Drift gây ra trên các biến (Features) mỏng manh
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# ĐẠI TU MODEL CNN-BiLSTM
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) 
        
        # Đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # CHUẨN HOÁ LAYER NORM THEO TỪNG TIME-STEP
        out = self.layer_norm(out)
        
        context_vector, attn_weights = self.attention(out)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out


In [15]:
class DynamicUndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, 
                 step=5, resample_each_epoch=False, rare_classes=None):
        

        if rare_classes is None:
            rare_classes = [0, 2, 3, 4, 5, 8, 10]
            
        feat_cols = [col for col in df.columns if col != 'label']
        self.X = torch.as_tensor(df[feat_cols].to_numpy(dtype=np.float32))
        self.y = torch.as_tensor(df['label'].to_numpy(dtype=np.int64))
        
        self.time_steps = time_steps
        self.step = step
        self.max_samples_per_class = max_samples_per_class
        self.resample_each_epoch = resample_each_epoch
        
        # Tạo 2 mảng index: 1 cái step=1 (bảo toàn), 1 cái có step (băm mỏng)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.class_indices = {}
        for c in np.unique(window_labels_step1):
            if c in rare_classes:
                # Bảo toàn 100% dữ liệu cho class hiếm
                self.class_indices[c] = all_indices_step1[window_labels_step1 == c]
            else:
                # Băm mỏng theo step đối với các class đa số
                self.class_indices[c] = all_indices_stepped[window_labels_stepped == c]
            
            print(f"Class {c}: Có sẵn {len(self.class_indices[c])} cửa sổ trong Pool")
            
        self._resample()
    
    def _resample(self):
        self.valid_indices = []
        for c, c_indices in self.class_indices.items():
            count = len(c_indices)
            
            # NẾU LÀ TẬP TRAIN (có bật resample_each_epoch)
            if self.resample_each_epoch:
                if count > self.max_samples_per_class:
                    # Dư thì băm đi (Undersampling)
                    sampled = np.random.choice(c_indices, self.max_samples_per_class, replace=False)
                else:
                    # Thiếu thì nhân bản (Oversampling)
                    sampled = np.random.choice(c_indices, self.max_samples_per_class, replace=True)
                    
            # NẾU LÀ TẬP VAL/TEST (Giữ nguyên gốc 100%, không thêm không bớt)
            else:
                sampled = c_indices
                
            self.valid_indices.extend(sampled)
            
        np.random.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices)
    def on_epoch_start(self):
        if self.resample_each_epoch:
            self._resample()

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        return window_X, label_y

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 5 
NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Dynamicsampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = DynamicUndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE, resample_each_epoch=True)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=1) 
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Dynamicsampling, set Train Step = 5)...
Class 0: Có sẵn 64488 cửa sổ trong Pool
Class 1: Có sẵn 314644 cửa sổ trong Pool
Class 2: Có sẵn 8164 cửa sổ trong Pool
Class 3: Có sẵn 117813 cửa sổ trong Pool
Class 4: Có sẵn 61666 cửa sổ trong Pool
Class 5: Có sẵn 7956 cửa sổ trong Pool
Class 6: Có sẵn 7698 cửa sổ trong Pool
Class 7: Có sẵn 69830 cửa sổ trong Pool
Class 8: Có sẵn 54420 cửa sổ trong Pool
Class 9: Có sẵn 26987 cửa sổ trong Pool
Class 10: Có sẵn 60328 cửa sổ trong Pool
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 14880 cửa sổ (step=1).
Class 1: Giữ nguyên 363049 cửa sổ (step=1).
Class 2: Giữ nguyên 1884 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giữ nguyên 27186 cửa sổ (step=1).
Class 4: Giữ nguyên 14229 cửa sổ (step=1).
Class 5: Giữ nguyên 1836 cửa sổ (step=1).
Class 6: Giữ nguyên 8880 cửa sổ (sử dụng step=1 để bảo toàn).
Clas

In [18]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    train_dataset.on_epoch_start() 
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 0.8521, Train Macro F1: 0.8909 | Val Loss: 0.6115, Val Macro F1: 0.9021


Epoch [2/30] - Train Loss: 0.6953, Train Macro F1: 0.9580 | Val Loss: 0.5916, Val Macro F1: 0.9214


Epoch [3/30] - Train Loss: 0.6788, Train Macro F1: 0.9653 | Val Loss: 0.5879, Val Macro F1: 0.9281


Epoch [4/30] - Train Loss: 0.6757, Train Macro F1: 0.9666 | Val Loss: 0.5862, Val Macro F1: 0.9195


...

Epoch [27/30] - Train Loss: 0.5963, Train Macro F1: 0.9970 | Val Loss: 0.5663, Val Macro F1: 0.9441


Epoch [28/30] - Train Loss: 0.5962, Train Macro F1: 0.9970 | Val Loss: 0.5661, Val Macro F1: 0.9456


Epoch [29/30] - Train Loss: 0.5969, Train Macro F1: 0.9967 | Val Loss: 0.5692, Val Macro F1: 0.9398


Epoch [30/30] - Train Loss: 0.5939, Train Macro F1: 0.9977 | Val Loss: 0.5649, Val Macro F1: 0.9455

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.8784    0.8185    0.8474     19846
           1     0.9989    1.0000    0.9995    484077
           2     0.2429    1.0000    0.3909      2515
           3     0.9982    0.8013    0.8890     36253
           4     0.6461    0.9932    0.7829     18979
           5     0.9815    0.9980    0.9897      2451
           6     0.7440    0.6987    0.7206     11847
           7     1.0000    0.9928    0.9964    107436
           8     0.8171    0.9955    0.8975     16746
           9     0.9993    0.6812    0.8101     41514
          10     0.9385    0.9876    0.9624     18567

    accuracy                         0.9621    760231
   macro avg     0.8404    0.9061    0.8442    760231
weighted avg     0.9751    0.9621    0.9645    760231



In [19]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    train_dataset.on_epoch_start() 
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 0.8737, Train Macro F1: 0.8829 | Val Loss: 0.6158, Val Macro F1: 0.8944


Epoch [2/30] - Train Loss: 0.7079, Train Macro F1: 0.9522 | Val Loss: 0.5994, Val Macro F1: 0.9018


Epoch [3/30] - Train Loss: 0.6880, Train Macro F1: 0.9618 | Val Loss: 0.5830, Val Macro F1: 0.9085


Epoch [4/30] - Train Loss: 0.6808, Train Macro F1: 0.9659 | Val Loss: 0.6084, Val Macro F1: 0.8842


...

Epoch [23/30] - Train Loss: 0.5936, Train Macro F1: 0.9976 | Val Loss: 0.5773, Val Macro F1: 0.9406


Epoch [24/30] - Train Loss: 0.5935, Train Macro F1: 0.9974 | Val Loss: 0.5746, Val Macro F1: 0.9434


Epoch [25/30] - Train Loss: 0.5933, Train Macro F1: 0.9976 | Val Loss: 0.5766, Val Macro F1: 0.9433


Epoch [26/30] - Train Loss: 0.5924, Train Macro F1: 0.9978 | Val Loss: 0.5773, Val Macro F1: 0.9426

[Early Stopping] Model không cải thiện sau 10 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.8844    0.9803    0.9299     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.2579    1.0000    0.4101      2515
           3     0.9999    0.7880    0.8814     36253
           4     0.6380    0.8766    0.7385     18979
           5     0.9843    0.9984    0.9913      2451
           6     0.7012    0.7121    0.7066     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.8397    0.9904    0.9089     16746
           9     0.9974    0.6831    0.8109     41514
          10     0.9332    0.9861    0.9589     18567

    accuracy                         0.9639    760231
   macro avg     0.8396    0.9105    0.8488    760231
weighted avg     0.9755    0.9639    0.9661    760231



Undersampling + CNN + LSTM + Attention + Sliding Window

In [20]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [15]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        # tạo mảng index cho step = 1 (dành riêng cho các class hiếm cần bảo toàn)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        # tạo mảng index theo step được truyền vào (cho các class còn lại)
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.valid_indices = []
        
        # lặp qua từng class (Lấy từ step=1 để đảm bảo không sót class nào)
        classes = np.unique(window_labels_step1)
        rng = np.random.default_rng(seed=42)
        for c in classes:
            # ƯU TIÊN GIỮ NGUYÊN băm cửa sổ 1-1 cho Class 2 và 6
            if c in [100]:
                c_indices = all_indices_step1[window_labels_step1 == c]
            else:
                c_indices = all_indices_stepped[window_labels_stepped == c]
                
            count = len(c_indices)
            
            # nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại
                 
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                if c in [2, 6]:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (sử dụng step=1 để bảo toàn).")
                else:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # lấy các index đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # lấy feature và label của cửa sổ trượt tại index thực sự
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 5 
NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = UndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=1) 
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, set Train Step = 5)...
Đang tính toán các cửa sổ hợp lệ (global step=5) và Undersampling...
Class 0: Giữ nguyên 12897 cửa sổ (step=5).
Class 1: Giảm từ 314644 xuống 20000 cửa sổ.
Class 2: Giữ nguyên 1633 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giảm từ 23563 xuống 20000 cửa sổ.
Class 4: Giữ nguyên 12333 cửa sổ (step=5).
Class 5: Giữ nguyên 1592 cửa sổ (step=5).
Class 6: Giữ nguyên 7698 cửa sổ (sử dụng step=1 để bảo toàn).
Class 7: Giảm từ 69830 xuống 20000 cửa sổ.
Class 8: Giữ nguyên 10884 cửa sổ (step=5).
Class 9: Giảm từ 26987 xuống 20000 cửa sổ.
Class 10: Giữ nguyên 12065 cửa sổ (step=5).
Tổng số cửa sổ sau khi lọc và Undersampling: 139102
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 14880 cửa sổ (step=1).
Class 1: Giữ nguyên 363049 cửa sổ (step=1).
Class 2: Giữ nguyên 1884 cửa sổ (sử dụng step=1 để bảo toàn).
Cla

In [23]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        # lớp Linear để tính điểm số (score) cho từng time-step
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        # đầu ra bi-lstm: (Batch, SeqLen, Hidden*2)
        
        # tính điểm chú tý cho mỗi timestap => (batch, seqlen, 1)
        scores = self.attention(lstm_outputs)
        
        # áp dung softmax để đưa sequence attention scores về dạng xác suất
        weights = torch.softmax(scores, dim=1)
        
        # nhân trọng số với output của lstm và tỉnh tổng (nhân chập)
        # (Batch, SeqLen, 1) * (Batch, SeqLen, Hidden*2) -> (Batch, Hidden*2)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        
        # trả lại context_vector và weights (để phục vụ cho việc trực quan hóa attention sau này)
        return context_vector, weights

# model CNN-BiLSTM với Cơ chế Attention
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # conv1d có padding=1 => giữ nguyên chiều dài chuỗi
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.dropout_cnn = nn.Dropout1d(0.2)

        # conv1d có padding=1 => giữ nguyên chiều dài chuỗi, tăng kênh lên 128
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        
        self.relu = nn.ReLU()
        
        # giảm 2 timesteps lại còn 1
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        # bi-lstm như cũ
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        

        # kích thước đầu vào của Attention = hidden_size * 2 (vì BiLSTM ghép 2 chiều ngược xuôi)
        self.attention = Attention(hidden_size * 2)
        
        self.dropout = nn.Dropout(0.5)
        
        # tầng phân loại cuối cùng
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # đầu vào (256,10, 314)
        
        # đảo trục cho CNN -> (256, 314, 10)
        x = x.permute(0, 2, 1) 
        
        # đi qua cnn block 1 => (256, 64, 10)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout_cnn(x)

        # đi qua cnn block 2 => (256, 128, 10)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        
        # giảm chiều dài chuỗi đi 2 lần => (256, 128, 5)
        x = self.pool(x)
        
        # đảo trục lại cho lstm (256, 5, 128)
        x = x.permute(0, 2, 1)
        
        # đi qua bi-lstm => (256, 5, 256)
        out, (hn, cn) = self.bilstm(x)
        
        # dùng attention để tính ra context vector: (256, 256)
        context_vector, attn_weights = self.attention(out)
        
        # đi qua lớp dense để ra dự đoán nhãn
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        # THAY ĐỔI DUY NHẤT Ở ĐÂY: in_channels = num_features (thay vì 1)
        # Conv1d sẽ trượt dọc theo chiều thời gian (Time_Steps) thay vì trượt dọc theo các features.
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        # tầng 2: giữ nguyên
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        # tầng 3: giữ nguyên
        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        # tầng BiLSTM: giữ nguyên
        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        # tầng attention: giữ nguyên
        self.attention = Attention(128 * 2) 

        # các tầng dense: giữ nguyên
        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        # tầng phân loại cuối cùng: giữ nguyên
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Đầu vào x từ SlidingWindowDataset đang có shape: (Batch, Time_Steps, Features)
        
        # THAY ĐỔI Ở ĐÂY: Dùng permute thay vì unsqueeze
        # Chuyển vị để shape trở thành: (Batch, Features, Time_Steps)
        # Để Conv1d quét qua chiều Time_Steps với số kênh tương ứng với số Features
        x = x.permute(0, 2, 1) 

        # đi qua 3 tầng Conv1dCNN (Code đoạn này giữ nguyên 100%)
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        # permute để đưa qua BiLSTM: (Batch, Channels, Time_Steps_New) -> (Batch, Time_Steps_New, Channels)
        # Chiều Channels lúc này đang là 128 (khớp hoàn hảo với input_size=128 của BiLSTM)
        x = x.permute(0, 2, 1)

        # đi qua BiLSTM
        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        # đi qua attention để lấy context vector
        context_vector, _ = self.attention(out)

        # đi qua các lớp dense
        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        return out

In [20]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 1.0459, Train Macro F1: 0.6246 | Val Loss: 0.6800, Val Macro F1: 0.6858


Epoch [2/30] - Train Loss: 0.7460, Train Macro F1: 0.8602 | Val Loss: 0.8655, Val Macro F1: 0.7780


Epoch [3/30] - Train Loss: 0.7067, Train Macro F1: 0.9218 | Val Loss: 0.6373, Val Macro F1: 0.8809


Epoch [4/30] - Train Loss: 0.6896, Train Macro F1: 0.9327 | Val Loss: 0.6628, Val Macro F1: 0.8187


...

Epoch [27/30] - Train Loss: 0.6021, Train Macro F1: 0.9793 | Val Loss: 0.5875, Val Macro F1: 0.9297


Epoch [28/30] - Train Loss: 0.6026, Train Macro F1: 0.9797 | Val Loss: 0.5885, Val Macro F1: 0.9317


Epoch [29/30] - Train Loss: 0.6012, Train Macro F1: 0.9798 | Val Loss: 0.5920, Val Macro F1: 0.9193


Epoch [30/30] - Train Loss: 0.6026, Train Macro F1: 0.9800 | Val Loss: 0.5896, Val Macro F1: 0.9232

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.8370    0.7793    0.8071     19846
           1     0.9999    1.0000    1.0000    484077
           2     0.3793    1.0000    0.5500      2515
           3     0.9998    0.8703    0.9306     36253
           4     0.4847    0.7499    0.5888     18979
           5     0.9967    0.9976    0.9971      2451
           6     0.8366    0.6961    0.7599     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.6794    0.9734    0.8003     16746
           9     0.9970    0.6723    0.8031     41514
          10     0.9986    0.9880    0.9933     18567

    accuracy                         0.9583    760231
   macro avg     0.8372    0.8843    0.8391    760231
weighted avg     0.9710    0.9583    0.9608    760231



In [21]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 1.0027, Train Macro F1: 0.6507 | Val Loss: 0.6417, Val Macro F1: 0.7227


Epoch [2/30] - Train Loss: 0.7372, Train Macro F1: 0.8340 | Val Loss: 0.6167, Val Macro F1: 0.7922


Epoch [3/30] - Train Loss: 0.7062, Train Macro F1: 0.8983 | Val Loss: 0.6969, Val Macro F1: 0.7824


Epoch [4/30] - Train Loss: 0.6921, Train Macro F1: 0.9310 | Val Loss: 0.8084, Val Macro F1: 0.6899


...

Epoch [27/30] - Train Loss: 0.5987, Train Macro F1: 0.9814 | Val Loss: 0.5864, Val Macro F1: 0.9303


Epoch [28/30] - Train Loss: 0.5982, Train Macro F1: 0.9815 | Val Loss: 0.5864, Val Macro F1: 0.9276


Epoch [29/30] - Train Loss: 0.5980, Train Macro F1: 0.9817 | Val Loss: 0.5880, Val Macro F1: 0.9317


Epoch [30/30] - Train Loss: 0.5960, Train Macro F1: 0.9834 | Val Loss: 0.5814, Val Macro F1: 0.9365

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.8905    0.7693    0.8254     19846
           1     0.9993    1.0000    0.9996    484077
           2     0.4057    0.9972    0.5768      2515
           3     0.9992    0.8818    0.9368     36253
           4     0.5057    0.8132    0.6236     18979
           5     0.9971    0.9967    0.9969      2451
           6     0.8420    0.7139    0.7726     11847
           7     0.9999    1.0000    1.0000    107436
           8     0.6864    0.9688    0.8035     16746
           9     0.9971    0.6733    0.8038     41514
          10     0.9984    0.9878    0.9931     18567

    accuracy                         0.9604    760231
   macro avg     0.8474    0.8911    0.8484    760231
weighted avg     0.9728    0.9604    0.9626    760231



In [24]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 0.9603, Train Macro F1: 0.7553 | Val Loss: 0.7338, Val Macro F1: 0.8429


Epoch [2/30] - Train Loss: 0.7223, Train Macro F1: 0.9334 | Val Loss: 0.6363, Val Macro F1: 0.8491


Epoch [3/30] - Train Loss: 0.6983, Train Macro F1: 0.9472 | Val Loss: 0.6413, Val Macro F1: 0.8722


Epoch [4/30] - Train Loss: 0.6843, Train Macro F1: 0.9535 | Val Loss: 0.6108, Val Macro F1: 0.8821


...

Epoch [27/30] - Train Loss: 0.6145, Train Macro F1: 0.9870 | Val Loss: 0.5735, Val Macro F1: 0.9388


Epoch [28/30] - Train Loss: 0.6142, Train Macro F1: 0.9867 | Val Loss: 0.5735, Val Macro F1: 0.9391


Epoch [29/30] - Train Loss: 0.6146, Train Macro F1: 0.9865 | Val Loss: 0.5734, Val Macro F1: 0.9397


Epoch [30/30] - Train Loss: 0.6137, Train Macro F1: 0.9866 | Val Loss: 0.5728, Val Macro F1: 0.9413

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.9344    0.7821    0.8515     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.4650    0.9984    0.6345      2515
           3     0.9998    0.9127    0.9543     36253
           4     0.5075    0.8372    0.6319     18979
           5     0.9984    0.9971    0.9978      2451
           6     0.8558    0.7084    0.7751     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.6878    0.9772    0.8073     16746
           9     1.0000    0.6725    0.8042     41514
          10     0.9986    0.9856    0.9921     18567

    accuracy                         0.9628    760231
   macro avg     0.8589    0.8974    0.8590    760231
weighted avg     0.9750    0.9628    0.9649    760231



In [ ]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 0.9504, Train Macro F1: 0.7562 | Val Loss: 0.6520, Val Macro F1: 0.8781


Epoch [2/30] - Train Loss: 0.7159, Train Macro F1: 0.9369 | Val Loss: 0.6105, Val Macro F1: 0.8886


Epoch [3/30] - Train Loss: 0.6936, Train Macro F1: 0.9490 | Val Loss: 0.6113, Val Macro F1: 0.9107


Epoch [4/30] - Train Loss: 0.6837, Train Macro F1: 0.9532 | Val Loss: 0.6123, Val Macro F1: 0.8874


...

Epoch [16/30] - Train Loss: 0.6246, Train Macro F1: 0.9816 | Val Loss: 0.5978, Val Macro F1: 0.9179


Epoch [17/30] - Train Loss: 0.6226, Train Macro F1: 0.9827 | Val Loss: 0.5881, Val Macro F1: 0.9288


Epoch [18/30] - Train Loss: 0.6231, Train Macro F1: 0.9823 | Val Loss: 0.6028, Val Macro F1: 0.9084


Epoch [19/30] - Train Loss: 0.6196, Train Macro F1: 0.9839 | Val Loss: 0.5899, Val Macro F1: 0.9242

[Early Stopping] Model không cải thiện sau 10 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.8116    0.7335    0.7706     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.3855    0.9988    0.5563      2515
           3     0.9998    0.8746    0.9331     36253
           4     0.4889    0.6841    0.5703     18979
           5     0.9878    0.9869    0.9873      2451
           6     0.7259    0.7645    0.7447     11847
           7     1.0000    1.0000    1.0000    107436
           8     0.6603    0.9861    0.7910     16746
           9     0.9999    0.6685    0.8013     41514
          10     0.9980    0.9834    0.9906     18567

    accuracy                         0.9566    760231
   macro avg     0.8234    0.8800    0.8314    760231
weighted avg     0.9684    0.9566    0.9589    760231



Undersampling + CNN + LSTM + Attention (attention + no sliding window) (cấu trúc giống paper gốc)

In [25]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_df.shape[1] - 1 # trừ đi cột label
NUM_CLASSES = len(train_df['label'].unique()) # số class
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        # tính điểm chú ý
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        # Nhân trọng số và tính tổng
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# kiến trúc chính của paper
class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        # tầng 1: conv1d với 128 filters, kernel size = 5, padding = 2 để giũ nguyên chiều dài chuỗi
        # in_channels = 1 (Vì ta coi 314 features như 1 chuỗi dài 314 đơn vị) (đưa từng flow riêng lẻ vào)
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        # tầng 2: conv1d với 256 filters, kernel size 5, ReLU -> Batch Norm, MaxPool(2), Dropout(0.3)
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        # tầng 3: conv1d với 128 filters, kernel size 3, ReLU -> Batch Norm, MaxPool(2), Dropout(0.3)
        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        # tầng BiLSTM: 128 units, Dropout(0.3)
        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        # tầng attention
        self.attention = Attention(128 * 2) # *2 vì là Bidirectional

        # các tầng dense
        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        # tầng phân loại cuối cùng
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Đầu vào x từ StandardFlowDataset đang có shape: (Batch, Features)
        # đầu vào x có shape: (256, 314)
        
        # shape trở thành: (256, 1, 314)
        x = x.unsqueeze(1) 

        # đi qua 3 tầng Conv1dCNN
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        # permute để đưa qua BiLSTM
        x = x.permute(0, 2, 1)

        # đi qua BiLSTM
        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        # đi qua attention để lấy context vector
        context_vector, _ = self.attention(out)

        # đi qua các lớp dense
        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        # không dùng softmax
        return out

# khởi tạo mô hình và đẩy lên GPU
model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)
print(model)

Paper_CNN_BiLSTM_Attention(
  (conv1): Conv1d(1, 128, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop1): Dropout1d(p=0.3, inplace=False)
  (conv2): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop2): Dropout1d(p=0.3, inplace=False)
  (conv3): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop3): Dropout1d(p=0.3, inplace=False)
  (bilstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (drop_lstm): Dropout(p=0.3, inp

In [27]:
from torch.utils.data import Dataset
import numpy as np
class StandardFlowDataset(Dataset):
    def __init__(self, df, max_samples_per_class=None):
        self.X = []
        self.y = []
        
        # nhóm theo nhãn và áp dụng undersampling
        for class_name, group in df.groupby('label'):
            if max_samples_per_class and len(group) > max_samples_per_class:
                group = group.sample(n=max_samples_per_class, random_state=42)
            
            self.X.append(group.drop(columns=['label']).values)
            self.y.append(group['label'].values)
            
        self.X = np.vstack(self.X)
        self.y = np.concatenate(self.y)
        
        # xáo trộn dữ liệu sau khi đã áp dụng undersampling để đảm bảo tính ngẫu nhiên
        idx = np.random.permutation(len(self.X))
        self.X = torch.tensor(self.X[idx], dtype=torch.float32)
        self.y = torch.tensor(self.y[idx], dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# khởi tạo dataset với max_samples_per_class để giới hạn số lượng mẫu của mỗi class trong tập train, còn val và test thì giữ nguyên để đánh giá chính xác hơn
MAX_SAMPLES = 20000 
train_dataset = StandardFlowDataset(train_df, max_samples_per_class=MAX_SAMPLES)
val_dataset = StandardFlowDataset(valid_df)
test_dataset = StandardFlowDataset(test_df) 

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [28]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 1.4410, Train Macro F1: 0.5408 | Val Loss: 0.7149, Val Macro F1: 0.6757


Epoch [2/30] - Train Loss: 1.1011, Train Macro F1: 0.7263 | Val Loss: 0.7030, Val Macro F1: 0.6927


Epoch [3/30] - Train Loss: 1.0416, Train Macro F1: 0.7517 | Val Loss: 0.6909, Val Macro F1: 0.7157


Epoch [4/30] - Train Loss: 0.9963, Train Macro F1: 0.7735 | Val Loss: 0.6887, Val Macro F1: 0.7260


...

Epoch [27/30] - Train Loss: 0.8775, Train Macro F1: 0.8289 | Val Loss: 0.6564, Val Macro F1: 0.7806


Epoch [28/30] - Train Loss: 0.8711, Train Macro F1: 0.8323 | Val Loss: 0.6553, Val Macro F1: 0.7865


Epoch [29/30] - Train Loss: 0.8702, Train Macro F1: 0.8332 | Val Loss: 0.6535, Val Macro F1: 0.7825


Epoch [30/30] - Train Loss: 0.8698, Train Macro F1: 0.8329 | Val Loss: 0.6557, Val Macro F1: 0.7817

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.5646    0.2318    0.3287     19846
           1     0.9988    1.0000    0.9994    484077
           2     0.4653    0.8052    0.5898      2515
           3     0.9757    0.8925    0.9322     36253
           4     0.5691    0.4875    0.5252     18979
           5     0.9091    0.9992    0.9520      2451
           6     0.4112    0.7020    0.5186     11847
           7     1.0000    0.9963    0.9982    107436
           8     0.4444    0.9214    0.5996     16746
           9     1.0000    0.6919    0.8179     41523
          10     0.7319    0.7978    0.7634     18567

    accuracy                         0.9327    760240
   macro avg     0.7336    0.7751    0.7295    760240
weighted avg     0.9459    0.9327    0.9332    760240



In [29]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 1.4331, Train Macro F1: 0.5399 | Val Loss: 0.7086, Val Macro F1: 0.6545


Epoch [2/30] - Train Loss: 1.0937, Train Macro F1: 0.7299 | Val Loss: 0.6966, Val Macro F1: 0.7027


Epoch [3/30] - Train Loss: 1.0400, Train Macro F1: 0.7544 | Val Loss: 0.6920, Val Macro F1: 0.6958


Epoch [4/30] - Train Loss: 1.0039, Train Macro F1: 0.7698 | Val Loss: 0.6818, Val Macro F1: 0.7315


...

Epoch [27/30] - Train Loss: 0.8730, Train Macro F1: 0.8283 | Val Loss: 0.6502, Val Macro F1: 0.7816


Epoch [28/30] - Train Loss: 0.8722, Train Macro F1: 0.8289 | Val Loss: 0.6507, Val Macro F1: 0.7774


Epoch [29/30] - Train Loss: 0.8711, Train Macro F1: 0.8298 | Val Loss: 0.6499, Val Macro F1: 0.7739


Epoch [30/30] - Train Loss: 0.8716, Train Macro F1: 0.8291 | Val Loss: 0.6500, Val Macro F1: 0.7728

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.6677    0.1914    0.2975     19846
           1     0.9999    1.0000    1.0000    484077
           2     0.5216    0.7956    0.6301      2515
           3     0.9811    0.9076    0.9429     36253
           4     0.5631    0.4013    0.4686     18979
           5     0.9662    0.9910    0.9784      2451
           6     0.5735    0.7004    0.6306     11847
           7     0.9999    0.9963    0.9981    107436
           8     0.4469    0.9078    0.5990     16746
           9     0.9998    0.6958    0.8205     41523
          10     0.4877    0.8564    0.6215     18567

    accuracy                         0.9315    760240
   macro avg     0.7461    0.7676    0.7261    760240
weighted avg     0.9464    0.9315    0.9305    760240



Hybrid Sampling (SMOTE + Undersampling) + CNN + LSTM + Attention + No Sliding Window

In [1]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [2]:
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from imblearn.combine import SMOTETomek
y_train = train_df['label']
x_train = train_df.drop(columns=['label'])
TARGET_COUNT = 100000
counts = y_train.value_counts().to_dict()



under_strategy = {label: TARGET_COUNT for label, count in counts.items() if count > TARGET_COUNT}
over_strategy = {label: TARGET_COUNT for label, count in counts.items() if count < TARGET_COUNT}

under = RandomUnderSampler(sampling_strategy=under_strategy, random_state=42)
smote_tomek = SMOTETomek(sampling_strategy=over_strategy, random_state=42, n_jobs=-1)

pipeline = Pipeline(steps=[('under', under), ('over', smote_tomek)])

print("Start to apply Hybrid Sampling (Under + Borderline SMOTE)...")
x_train_resampled, y_train_resampled = pipeline.fit_resample(x_train, y_train)

print(f"Number of samples after resampling: {len(y_train_resampled)}")
print(y_train_resampled.value_counts())
train_resampled_df = pd.DataFrame(x_train_resampled, columns=x_train.columns)
train_resampled_df['label'] = y_train_resampled

Start to apply Hybrid Sampling (Under + Borderline SMOTE)...
Number of samples after resampling: 1079338
label
2     100000
7     100000
1      99998
9      99997
5      99996
3      99912
6      99645
8      98665
10     93963
4      93770
0      93392
Name: count, dtype: int64


C:\Users\Admin\AppData\Local\Temp\ipykernel_15708\653627142.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_resampled_df['label'] = y_train_resampled


In [22]:
from torch.utils.data import Dataset
import torch
import numpy as np
from torch.utils.data import DataLoader
class StandardFlowDataset(Dataset):
    def __init__(self, df, max_samples_per_class=None):
        self.X = []
        self.y = []
        
        for class_name, group in df.groupby('label'):
            if max_samples_per_class and len(group) > max_samples_per_class:
                group = group.sample(n=max_samples_per_class, random_state=42)
            
            self.X.append(group.drop(columns=['label']).values)
            self.y.append(group['label'].values)
            
        self.X = np.vstack(self.X)
        self.y = np.concatenate(self.y)
        
        idx = np.random.permutation(len(self.X))
        self.X = torch.tensor(self.X[idx], dtype=torch.float32)
        self.y = torch.tensor(self.y[idx], dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [23]:
BATCH_SIZE = 512
train_dataset = StandardFlowDataset(train_resampled_df)
val_dataset = StandardFlowDataset(valid_df)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_dataset = StandardFlowDataset(test_df)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_dataset.X.shape[1]
NUM_CLASSES = len(train_dataset.y.unique())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        self.attention = Attention(128 * 2) 

        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        
        x = x.unsqueeze(1) 

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        x = x.permute(0, 2, 1)

        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        context_vector, _ = self.attention(out)

        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        return out


In [7]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
import torch.optim as optim

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 1.1169, Train Macro F1: 0.6881 | Val Loss: 0.6937, Val Macro F1: 0.7352


Epoch [2/30] - Train Loss: 0.9283, Train Macro F1: 0.7824 | Val Loss: 0.6754, Val Macro F1: 0.7546


Epoch [3/30] - Train Loss: 0.8822, Train Macro F1: 0.8083 | Val Loss: 0.6656, Val Macro F1: 0.7625


Epoch [4/30] - Train Loss: 0.8654, Train Macro F1: 0.8177 | Val Loss: 0.6567, Val Macro F1: 0.7793


...

Epoch [27/30] - Train Loss: 0.8054, Train Macro F1: 0.8781 | Val Loss: 0.6444, Val Macro F1: 0.7871


Epoch [28/30] - Train Loss: 0.8044, Train Macro F1: 0.8786 | Val Loss: 0.6558, Val Macro F1: 0.7708


Epoch [29/30] - Train Loss: 0.8058, Train Macro F1: 0.8784 | Val Loss: 0.6435, Val Macro F1: 0.7833


Epoch [30/30] - Train Loss: 0.7951, Train Macro F1: 0.8836 | Val Loss: 0.6366, Val Macro F1: 0.8132

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.7880    0.6806    0.7304     19846
           1     1.0000    0.9999    0.9999    484077
           2     0.4146    0.9571    0.5785      2515
           3     0.9903    0.8573    0.9190     36253
           4     0.6265    0.8335    0.7154     18979
           5     0.6831    0.9992    0.8115      2451
           6     0.5960    0.7157    0.6504     11847
           7     1.0000    0.9963    0.9982    107436
           8     0.9727    0.9074    0.9389     16746
           9     1.0000    0.6653    0.7990     41523
          10     0.5373    0.8257    0.6510     18567

    accuracy                         0.9510    760240
   macro avg     0.7826    0.8580    0.7993    760240
weighted avg     0.9635    0.9510    0.9534    760240



In [25]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 10

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_1.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_1.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 1.0978, Train Macro F1: 0.6976 | Val Loss: 0.6845, Val Macro F1: 0.7237


Epoch [2/30] - Train Loss: 0.9063, Train Macro F1: 0.7962 | Val Loss: 0.6686, Val Macro F1: 0.7604


Epoch [3/30] - Train Loss: 0.8798, Train Macro F1: 0.8083 | Val Loss: 0.6712, Val Macro F1: 0.7363


Epoch [4/30] - Train Loss: 0.8676, Train Macro F1: 0.8144 | Val Loss: 0.6607, Val Macro F1: 0.7526


...

Epoch [27/30] - Train Loss: 0.8174, Train Macro F1: 0.8686 | Val Loss: 0.6405, Val Macro F1: 0.8098


Epoch [28/30] - Train Loss: 0.8168, Train Macro F1: 0.8695 | Val Loss: 0.6433, Val Macro F1: 0.8118


Epoch [29/30] - Train Loss: 0.8043, Train Macro F1: 0.8757 | Val Loss: 0.6359, Val Macro F1: 0.8482


Epoch [30/30] - Train Loss: 0.8016, Train Macro F1: 0.8772 | Val Loss: 0.6459, Val Macro F1: 0.7895

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST Residual CNN LSTM No Attention ---


              precision    recall  f1-score   support

           0     0.7817    0.6840    0.7296     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.3590    0.9618    0.5229      2515
           3     0.9890    0.8594    0.9196     36253
           4     0.6114    0.8684    0.7176     18979
           5     0.9566    0.9984    0.9770      2451
           6     0.5889    0.7014    0.6403     11847
           7     0.9985    0.9963    0.9974    107436
           8     0.9655    0.9012    0.9322     16746
           9     1.0000    0.6730    0.8045     41523
          10     0.5671    0.7986    0.6632     18567

    accuracy                         0.9515    760240
   macro avg     0.8016    0.8584    0.8095    760240
weighted avg     0.9639    0.9515    0.9540    760240

